In [32]:
import pandas as pd
import numpy as np
from IPython.display import display, HTML
import folium
from folium.plugins import HeatMap
from shapely.geometry import Point, MultiPoint
import re

In [33]:
# Paths and read configs for each source
DATA_DIR = "../dados"

SOURCES = {
    "Ocorrências Criminais": {
        "path": f"{DATA_DIR}/df_ocorrencias_tratado - Extração 1 .csv",
        "encoding": "utf-8",
        "sep": ",",
        "desc": "Registros de furtos e roubos com lat/long — base da mancha criminal.",
    },
    "Disque Denúncia": {
        "path": f"{DATA_DIR}/disk_denuncia.csv",
        "encoding": "latin1",
        "sep": ";",
        "desc": "Denúncias em texto livre — dinâmica criminal, rotinas e suspeitos.",
    },
    "Fatores Urbanos": {
        "path": f"{DATA_DIR}/fatores_urbanos.csv",
        "encoding": "utf-8",
        "sep": ",",
        "desc": "Fatores ambientais de incidência: iluminação, obstrução, mobiliário etc.",
    },
    "Câmeras / Polígonos FM": {
        "path": f"{DATA_DIR}/cameras_areas_fm.csv",
        "encoding": "utf-8",
        "sep": ",",
        "desc": "Pontos de câmera e trechos das áreas operacionais da Força Municipal.",
    },
    "Domínio Territorial": {
        "path": f"{DATA_DIR}/outros dados/dominio_territorial - Extração 1.csv",
        "encoding": "utf-8",
        "sep": ",",
        "desc": "Controle territorial por ORCRIM — camada de dinâmica criminal.",
    },
}

In [34]:
def preview_dataset(name, cfg):
    """Load and display a full preview of one dataset."""
    df = pd.read_csv(cfg["path"], encoding=cfg["encoding"], sep=cfg["sep"], low_memory=False)

    display(HTML(f"<h2 style='color:#1a4e8a;border-bottom:2px solid #1a4e8a;padding-bottom:4px'>{name}</h2>"))
    display(HTML(f"<p><em>{cfg['desc']}</em></p>"))

    n_rows, n_cols = df.shape
    missing = df.isnull().sum()
    missing_pct = (missing / n_rows * 100).round(1)
    schema = pd.DataFrame({
        "dtype": df.dtypes,
        "n_missing": missing,
        "pct_missing": missing_pct,
        "n_unique": df.nunique(),
        "example": df.iloc[0] if len(df) > 0 else pd.Series(dtype=object),
    })

    display(HTML(f"<b>Shape:</b> {n_rows:,} linhas × {n_cols} colunas"))
    display(HTML("<b>Schema:</b>"))
    display(schema.style.background_gradient(subset=["pct_missing"], cmap="Oranges"))

    display(HTML("<b>Primeiras 5 linhas:</b>"))
    display(df.head())

    num_cols = df.select_dtypes(include="number").columns.tolist()
    if num_cols:
        display(HTML("<b>Estatísticas descritivas (colunas numéricas):</b>"))
        display(df[num_cols].describe().round(3))

    print()
    return df

In [35]:
# ── Ocorrências Criminais ──────────────────────────────────────────────────────
df_ocorrencias = preview_dataset("Ocorrências Criminais", SOURCES["Ocorrências Criminais"])

,dtype,n_missing,pct_missing,n_unique,example
id_criptografado,object,0,0.000000,114252,0259749088cab44c49ca762835e830a0edd6fedb64ce14a68a91f9776f181213
ano,int64,0,0.000000,5,2023
data,object,22,0.000000,2110,nan
mes,int64,0,0.000000,12,5
hora,object,22,0.000000,1436,nan
delito,int64,0,0.000000,3,15
longitude,float64,0,0.000000,66053,-43.267676
latitude,float64,0,0.000000,65868,-22.862878
desc_delito,object,0,0.000000,3,Roubo a transeunte
aisp,int64,0,0.000000,17,16


,id_criptografado,ano,data,mes,hora,delito,longitude,latitude,desc_delito,aisp,risp,locf,dia_semana,geometria
0,0259749088cab44c49ca762835e830a0edd6fedb64ce14...,2023,NaN,5,NaN,15,-43.267676,-22.862878,Roubo a transeunte,16,1,Rua arapa,NaN,POINT(-43.26767616 -22.86287837)
1,70af7ad098e687d1c8672eb108709ac2521a5fa763622a...,2021,NaN,12,NaN,15,-43.177214,-22.908850,Roubo a transeunte,5,1,Avenida treze de maio,NaN,POINT(-43.17721362 -22.90884995)
2,69abf67716dbf941722a0cc66145629b358a0bca240586...,2022,NaN,8,NaN,15,-43.178471,-22.964581,Roubo a transeunte,19,1,Rua duvivier,NaN,POINT(-43.17847063 -22.96458062)
3,89bdd094dbf38a61e46f5b753d1ac7d8e06cac9db8f43c...,2022,NaN,3,NaN,15,-43.219676,-22.983867,Roubo a transeunte,23,1,Avenida ataulfo de paiva,NaN,POINT(-43.21967599 -22.98386735)
4,ce89e031b4ce0f08cfdf30b5d74bfca7deaa95e0ab2c3d...,2023,NaN,4,NaN,15,-43.354368,-22.803981,Roubo a transeunte,41,2,Avenida sargento de milicias,NaN,POINT(-43.35436847 -22.80398115)


,ano,mes,delito,longitude,latitude,aisp,risp
count,115354.000,115354.000,115354.000,115354.000,115354.000,115354.000,115354.000
mean,2021.858,6.342,16.262,-46.685,-28.232,15.453,1.465
std,1.441,3.511,1.770,382.137,348.727,12.528,0.499
min,2020.000,1.000,15.000,-43466.000,-22901.000,2.000,1.000
25%,2021.000,3.000,15.000,-43.362,-22.920,5.000,1.000
50%,2022.000,6.000,15.000,-43.302,-22.896,14.000,1.000
75%,2023.000,9.000,19.000,-43.224,-22.863,22.000,2.000
max,2024.000,12.000,19.000,-43.159,-22.784,41.000,2.000


In [36]:
# ── Disque Denúncia ───────────────────────────────────────────────────────────
df_dd = preview_dataset("Disque Denúncia", SOURCES["Disque Denúncia"])

,dtype,n_missing,pct_missing,n_unique,example
numero_denuncia,object,65546,78.500000,18003,1024.6.2020
id_denuncia,float64,65546,78.500000,18003,2301680.000000
data_denuncia,object,65546,78.500000,17920,6/4/2020 8:16:00
data_difusao,object,65546,78.500000,9649,6/15/2020 14:57:00
tipo_logradouro,object,65558,78.500000,43,R
logradouro,object,65546,78.500000,5510,SANTO CRISTO
numero_logradouro,object,76511,91.600000,1460,nan
complemento_logradouro,object,81079,97.000000,1189,nan
bairro_logradouro,object,65546,78.500000,343,SANTO CRISTO
subbairro_logradouro,object,79732,95.400000,176,nan


,numero_denuncia,id_denuncia,data_denuncia,data_difusao,tipo_logradouro,logradouro,numero_logradouro,complemento_logradouro,bairro_logradouro,subbairro_logradouro,...,timestamp_insercao,id_classe,classe,tipos.id_tipo,tipos.tipo,tipos.assunto_principal,id_tipo,tipo,assunto_principal,relato_redacted
0,1024.6.2020,2301680.0,6/4/2020 8:16:00,6/15/2020 14:57:00,R,SANTO CRISTO,NaN,NaN,SANTO CRISTO,NaN,...,7/15/2024 0:00:00,12.0,SUBSTÂNCIAS ENTORPECENTES,84.0,CONSUMO DE DROGAS,1.0,84.0,CONSUMO DE DROGAS,1.0,NA RUA CITADA ESQUINA COM A VIA BINARIO DO [NO...
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,id_denuncia,cep_logradouro,xptos.id,orgaos.id,assuntos.id_classe,assuntos.tipos.id_tipo,assuntos.tipos.assunto_principal,envolvidos.id,envolvidos.nome,envolvidos.vulgo,envolvidos.idade,id_classe,tipos.id_tipo,tipos.assunto_principal,id_tipo,assunto_principal
count,18003.000,1.800300e+04,4268.000,81613.000,36012.000,41761.000,41761.000,4399.000,0.0,0.0,4399.000,18003.000,21754.000,21754.000,18003.000,18003.000
mean,2497079.869,1.740026e+07,321.030,916.227,7.094,61.216,0.431,1154271.155,NaN,NaN,33.354,7.129,59.101,0.538,59.177,0.558
std,199156.183,8.651756e+06,105.121,656.683,4.589,37.278,0.495,155622.645,NaN,NaN,12.593,4.998,36.236,0.499,33.881,0.497
min,2163866.000,0.000000e+00,2.000,14.000,1.000,1.000,0.000,903553.000,NaN,NaN,0.000,2.000,12.000,0.000,20.000,0.000
25%,2332352.000,2.024120e+07,304.000,347.000,2.000,20.000,0.000,1023668.000,NaN,NaN,24.000,2.000,20.000,0.000,20.000,0.000
50%,2483880.000,2.125059e+07,363.000,497.000,8.000,77.000,0.000,1142721.000,NaN,NaN,30.000,12.000,84.000,1.000,84.000,1.000
75%,2658214.500,2.204108e+07,387.000,1584.000,12.000,84.000,1.000,1265386.500,NaN,NaN,40.000,12.000,84.000,1.000,84.000,1.000
max,2874136.000,2.896851e+07,423.000,1942.000,19.000,187.000,1.000,1455430.000,NaN,NaN,88.000,12.000,164.000,1.000,114.000,1.000


In [37]:
# ── Fatores Urbanos ───────────────────────────────────────────────────────────
df_fatores = preview_dataset("Fatores Urbanos", SOURCES["Fatores Urbanos"])

,dtype,n_missing,pct_missing,n_unique,example
id_resposta_ocorrencia,int64,0,0.000000,2085,732
logradouro,object,0,0.000000,298,Rua Coração de Maria
numero_porta,object,183,8.800000,682,426
referencia,object,904,43.400000,832,nan
coordenada_x,float64,0,0.000000,1672,-22.891580
coordenada_y,float64,0,0.000000,1676,-43.274256
observacao,object,977,46.900000,866,nan
endereco_informado,bool,0,0.000000,2,False
valido,object,821,39.400000,1,nan
id_bairro,int64,0,0.000000,48,63


,id_resposta_ocorrencia,logradouro,numero_porta,referencia,coordenada_x,coordenada_y,observacao,endereco_informado,valido,id_bairro,...,id_item_praca,item_praca_descricao,id_tipo_ocorrencia,tipo_ocorrencia_descricao,tipo_ocorrencia_ativo,orgao_responsavel,ocorrencia_informacao,id_orgao_ocorrencia,ocorrencia_orgao_nome,codigo_ocorrencia_orgao
0,732,Rua Coração de Maria,426,NaN,-22.891580,-43.274256,NaN,False,NaN,63,...,NaN,NaN,5,Vegetação obstruindo a visibilidade do passeio,True,COMLURB,O agente deve identificar se há presença de ve...,1,COMLURB,4351
1,734,Rua Coração de Maria,406,NaN,-22.891878,-43.274529,NaN,False,NaN,63,...,NaN,NaN,5,Vegetação obstruindo a visibilidade do passeio,True,COMLURB,O agente deve identificar se há presença de ve...,1,COMLURB,4351
2,705,Rua Cabrália,16,NaN,-22.858458,-43.374460,TRÊS PODAS,False,NaN,90,...,NaN,NaN,5,Vegetação obstruindo a visibilidade do passeio,True,COMLURB,O agente deve identificar se há presença de ve...,1,COMLURB,4351
3,729,Rua Miguel Fernandes,543,NaN,-22.897172,-43.268882,NaN,False,NaN,63,...,NaN,NaN,5,Vegetação obstruindo a visibilidade do passeio,True,COMLURB,O agente deve identificar se há presença de ve...,1,COMLURB,4351
4,747,Rua Silva Mourão,70,NaN,-22.888777,-43.279309,NaN,False,NaN,65,...,NaN,NaN,5,Vegetação obstruindo a visibilidade do passeio,True,COMLURB,O agente deve identificar se há presença de ve...,1,COMLURB,4351


,id_resposta_ocorrencia,coordenada_x,coordenada_y,id_bairro,id_tipo_pessoa,id_subarea,id_ocupacao_pessoa,id_tipo_frequencia,ocupacao_drogas,id_item_praca,id_tipo_ocorrencia,id_orgao_ocorrencia,codigo_ocorrencia_orgao
count,2085.000,2085.000,2085.000,2085.000,0.0,1867.000,0.0,0.0,0.0,0.0,2085.000,2085.000,2085.000
mean,1819.202,-22.928,-43.270,52.474,NaN,12.258,NaN,NaN,NaN,NaN,18.290,4.093,3227.219
std,918.472,0.042,0.112,41.497,NaN,5.212,NaN,NaN,NaN,NaN,8.976,3.432,1256.084
min,237.000,-23.011,-43.684,3.000,NaN,0.000,NaN,NaN,NaN,NaN,5.000,1.000,1.000
25%,1026.000,-22.966,-43.323,20.000,NaN,8.000,NaN,NaN,NaN,NaN,12.000,1.000,1700.000
50%,1692.000,-22.919,-43.221,33.000,NaN,13.000,NaN,NaN,NaN,NaN,19.000,4.000,3900.000
75%,2659.000,-22.894,-43.190,83.000,NaN,15.000,NaN,NaN,NaN,NaN,26.000,6.000,4351.000
max,3419.000,-22.822,-43.169,149.000,NaN,22.000,NaN,NaN,NaN,NaN,41.000,19.000,4351.000


In [38]:
# ── Câmeras / Polígonos FM ────────────────────────────────────────────────────
df_cameras = preview_dataset("Câmeras / Polígonos FM", SOURCES["Câmeras / Polígonos FM"])

,dtype,n_missing,pct_missing,n_unique,example
id_ponto,object,0,0.000000,985,8f30106e-358f-4e8f-b94a-dc748e9624a9
nome_area_fm,object,0,0.000000,9,Presidente Vargas - Campo de Santana - Central do Brasil - Cinelândia
id_trecho,int64,0,0.000000,777,203724
geometry,object,0,0.000000,985,POINT (-43.18021712228354 -22.909370871507164)


,id_ponto,nome_area_fm,id_trecho,geometry
0,8f30106e-358f-4e8f-b94a-dc748e9624a9,Presidente Vargas - Campo de Santana - Central...,203724,POINT (-43.18021712228354 -22.909370871507164)
1,e1d1a5e7-159f-487b-91df-6d660e6b5cba,Presidente Vargas - Campo de Santana - Central...,46564,POINT (-43.18233018600572 -22.904098547395474)
2,39623659-3247-4232-aca8-d885c9d10181,Presidente Vargas - Campo de Santana - Central...,306043,POINT (-43.180528574077265 -22.901827901759894)
3,62b1d57c-1085-44f9-8fa4-f0225cbc96e4,Presidente Vargas - Campo de Santana - Central...,70354,POINT (-43.17651942254311 -22.91051860226564)
4,dfabd902-d188-4a6d-b983-503c08c35836,Presidente Vargas - Campo de Santana - Central...,51517,POINT (-43.18307050654744 -22.904831290621335)


,id_trecho
count,985.000
mean,129077.201
std,78728.378
min,16703.000
25%,71089.000
50%,97987.000
75%,201399.000
max,306044.000


In [39]:
# ── Domínio Territorial ───────────────────────────────────────────────────────
df_dominio = preview_dataset("Domínio Territorial", SOURCES["Domínio Territorial"])

,dtype,n_missing,pct_missing,n_unique,example
nome_territorio,object,0,0.000000,1548,MORRO SANTANA
dominio_orcrim,object,0,0.000000,4,ADA
geometria,object,0,0.000000,1628,"POLYGON((-41.7902341999999 -22.3715415999999, -41.7890968999999 -22.3724941, -41.7887964999999 -22.3736648, -41.7864577 -22.3743393999999, -41.7861786999999 -22.3720432, -41.7856207999999 -22.3718843999999, -41.7858138999999 -22.3707479, -41.7860499999999 -22.3697955, -41.7882600999999 -22.3692397999999, -41.7898264999999 -22.3698747999999, -41.7904273 -22.3699541999999, -41.7902341999999 -22.3715415999999))"


,nome_territorio,dominio_orcrim,geometria
0,MORRO SANTANA,ADA,"POLYGON((-41.7902341999999 -22.3715415999999, ..."
1,MORRO DO CARVÃO,ADA,"POLYGON((-41.7751943 -22.3900022999999, -41.77..."
2,PARQUE SAO MATEUS,ADA,"POLYGON((-41.3102325 -21.7165644, -41.30911669..."
3,FAVELA DAS ALMAS,ADA,"POLYGON((-43.4496900999999 -22.8866939, -43.44..."
4,MIRAMAR,ADA,"POLYGON((-41.7912426999999 -22.3807538, -41.79..."


In [40]:
# ── Resumo consolidado das 5 fontes ──────────────────────────────────────────
summary_data = {}
for name, cfg in SOURCES.items():
    df = pd.read_csv(cfg["path"], encoding=cfg["encoding"], sep=cfg["sep"], low_memory=False)
    summary_data[name] = {
        "Linhas": f"{len(df):,}",
        "Colunas": len(df.columns),
        "% missing (média)": f"{df.isnull().mean().mean()*100:.1f}%",
        "Encoding": cfg["encoding"],
        "Separador": repr(cfg["sep"]),
    }

display(HTML("<h2 style='color:#1a4e8a'>Resumo das 5 fontes</h2>"))
display(pd.DataFrame(summary_data).T)

,Linhas,Colunas,% missing (média),Encoding,Separador
Ocorrências Criminais,"115,354",14,0.0%,utf-8,"','"
Disque Denúncia,"83,549",48,77.2%,latin1,';'
Fatores Urbanos,"2,085",31,36.9%,utf-8,"','"
Câmeras / Polígonos FM,985,4,0.0%,utf-8,"','"
Domínio Territorial,"1,628",3,0.0%,utf-8,"','"


---
## Mapa de segmentos quentes — Rua Lauro Müller / Av. General Severiano / Av. Venceslau Brás

Replicação do mapa CompStat com dados reais: heatmap criminal 2023-2024, polígono da área FM, câmeras e segmentos críticos coloridos por densidade de crime (crimes em raio de ~150 m de cada câmera/trecho).

In [41]:
# ── Parâmetros da área de interesse ──────────────────────────────────────────
AREA_NAME  = "Rua Lauro Müller – Avenida General Severiano – Avenida Venceslau Brás"
ANOS       = [2023, 2024]

# Bounding box da área (ligeiramente maior que o convex hull das câmeras)
BB = dict(lat_min=-22.975, lat_max=-22.945, lon_min=-43.190, lon_max=-43.165)

# Raio em graus para contar crimes próximos a cada câmera (~150 m no Rio)
BUFFER_DEG = 0.0015

# Paleta dos segmentos críticos — espelha a legenda do print de referência
BINS   = [0,  11,  14,  17,  20,  23,  np.inf]
COLORS = ["#cccccc", "#f4a636", "#e07b2a", "#c85020", "#a02010", "#700000"]

In [42]:
# ── 1. Ocorrências 2023-2024 filtradas para a bbox ────────────────────────────
df_oc_full = pd.read_csv(
    SOURCES["Ocorrências Criminais"]["path"], encoding="utf-8", low_memory=False
)

df_crimes = df_oc_full[
    df_oc_full["ano"].isin(ANOS)
    & df_oc_full["latitude"].between(BB["lat_min"], BB["lat_max"])
    & df_oc_full["longitude"].between(BB["lon_min"], BB["lon_max"])
].copy()

print(f"Crimes 2023-2024 na área: {len(df_crimes):,}")
print(df_crimes["desc_delito"].value_counts())

Crimes 2023-2024 na área: 1,832
desc_delito
Roubo a transeunte           1036
Roubo de aparelho celular     734
Roubo em coletivo              62
Name: count, dtype: int64


In [43]:
# ── 2. Câmeras da área e extração de lat/lon ──────────────────────────────────
df_cam_full = pd.read_csv(
    SOURCES["Câmeras / Polígonos FM"]["path"], encoding="utf-8", low_memory=False
)
df_cam_area = df_cam_full[df_cam_full["nome_area_fm"] == AREA_NAME].copy()

coords_parsed = df_cam_area["geometry"].str.extract(r"POINT \(([^ ]+) ([^\)]+)\)")
df_cam_area["lon"] = coords_parsed[0].astype(float)
df_cam_area["lat"] = coords_parsed[1].astype(float)

print(f"Câmeras na área: {len(df_cam_area)} | Trechos únicos: {df_cam_area['id_trecho'].nunique()}")

Câmeras na área: 50 | Trechos únicos: 42


In [44]:
# ── 3. Polígono operacional = convex hull das câmeras ─────────────────────────
cam_points = MultiPoint([Point(r.lon, r.lat) for _, r in df_cam_area.iterrows()])
hull_coords = [
    (lat, lon) for lon, lat in cam_points.convex_hull.exterior.coords
]

In [45]:
# ── 4. Score de crimes por câmera (contagem dentro do buffer) ─────────────────
crime_arr = df_crimes[["latitude", "longitude"]].to_numpy()

def crimes_nearby(cam_lat, cam_lon, radius=BUFFER_DEG):
    dist = np.sqrt((crime_arr[:, 0] - cam_lat) ** 2 + (crime_arr[:, 1] - cam_lon) ** 2)
    return int((dist <= radius).sum())

df_cam_area["n_crimes"] = [
    crimes_nearby(r.lat, r.lon) for _, r in df_cam_area.iterrows()
]

def bin_color(n):
    for i in range(len(BINS) - 1):
        if BINS[i] <= n < BINS[i + 1]:
            return COLORS[i]
    return COLORS[-1]

df_cam_area["seg_color"] = df_cam_area["n_crimes"].apply(bin_color)

print(df_cam_area["n_crimes"].describe().round(1))
print("\nDistribuição por faixa:")
print(pd.cut(df_cam_area["n_crimes"], bins=BINS, right=False).value_counts().sort_index())

count    50.0
mean     25.2
std      11.8
min       4.0
25%      16.0
50%      30.0
75%      33.0
max      43.0
Name: n_crimes, dtype: float64

Distribuição por faixa:
n_crimes
[0.0, 11.0)      9
[11.0, 14.0)     1
[14.0, 17.0)     5
[17.0, 20.0)     1
[20.0, 23.0)     1
[23.0, inf)     33
Name: count, dtype: int64


In [ ]:
# ── 5. Construir o mapa ───────────────────────────────────────────────────────
import os

center_lat = (BB["lat_min"] + BB["lat_max"]) / 2
center_lon = (BB["lon_min"] + BB["lon_max"]) / 2

m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=15,
    tiles="CartoDB positron",
)

# --- Polígono da área operacional (azul) ---
folium.Polygon(
    locations=hull_coords,
    color="#1a5fad",
    weight=2,
    fill=True,
    fill_color="#4a8fd4",
    fill_opacity=0.12,
    tooltip=AREA_NAME,
).add_to(m)

# --- Heatmap de crimes ---
crime_pts = df_crimes[["latitude", "longitude"]].values.tolist()
HeatMap(
    crime_pts,
    radius=18,
    blur=22,
    min_opacity=0.3,
    gradient={0.2: "#fff5f0", 0.5: "#fc8d59", 0.8: "#d7301f", 1.0: "#7f0000"},
    name="Heatmap crimes 2023-2024",
).add_to(m)

# --- Segmentos críticos coloridos por n_crimes ---
seg_layer = folium.FeatureGroup(name="Segmentos críticos", show=True)
for _, row in df_cam_area[df_cam_area["n_crimes"] >= 11].iterrows():
    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=11,
        color=row["seg_color"],
        fill=True,
        fill_color=row["seg_color"],
        fill_opacity=0.85,
        weight=2,
        tooltip=f"Trecho {row['id_trecho']} — {row['n_crimes']} crimes (150 m)",
    ).add_to(seg_layer)
seg_layer.add_to(m)

# --- Câmeras (pontos azuis pequenos) ---
cam_layer = folium.FeatureGroup(name="Câmeras da cidade", show=True)
for _, row in df_cam_area.iterrows():
    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=4,
        color="#1a5fad",
        fill=True,
        fill_color="#4a8fd4",
        fill_opacity=0.85,
        weight=1,
        tooltip=f"Câmera | Trecho {row['id_trecho']} | {row['n_crimes']} crimes",
    ).add_to(cam_layer)
cam_layer.add_to(m)

# --- Legenda ---
legend_html = """
<div style="position:fixed;bottom:40px;right:20px;z-index:1000;
            background:white;padding:14px 18px;border-radius:8px;
            border:1px solid #ccc;font-family:Arial,sans-serif;font-size:13px;
            box-shadow:2px 2px 6px rgba(0,0,0,.2);">
  <b>Legenda:</b><br><br>
  <span style="color:#1a5fad">&#9679;</span> Câmera da cidade<br>
  <br><b>Segmentos críticos (crimes ≈150 m):</b><br>
  <span style="color:#f4a636">&#9679;</span> 11 – 14<br>
  <span style="color:#e07b2a">&#9679;</span> 14 – 17<br>
  <span style="color:#c85020">&#9679;</span> 17 – 20<br>
  <span style="color:#a02010">&#9679;</span> 20 – 23<br>
  <span style="color:#700000">&#9679;</span> 23 +<br>
  <br><b>Crimes 2023-2024:</b><br>
  <span style="background:linear-gradient(to right,#fff5f0,#d7301f,#7f0000);
    display:inline-block;width:80px;height:10px;border-radius:3px"></span><br>
  Mínimo &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; Máximo
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

# --- Título ---
title_html = f"""
<div style="position:fixed;top:12px;left:50%;transform:translateX(-50%);
            z-index:1000;background:white;padding:8px 16px;border-radius:6px;
            border:1px solid #ccc;font-family:Arial,sans-serif;font-size:13px;
            box-shadow:2px 2px 6px rgba(0,0,0,.2);max-width:640px;text-align:center;">
  <b>Mapa de segmentos quentes — {AREA_NAME}</b><br>
  <span style="color:#555">Total de roubos: {len(df_crimes):,} ocorrências, 2023-2024</span>
</div>
"""
m.get_root().html.add_child(folium.Element(title_html))

folium.LayerControl().add_to(m)

# Salva HTML para abrir no browser
os.makedirs("../output", exist_ok=True)
m.save("../output/mapa_lauro_muller.html")
print("Mapa salvo → abra: output/mapa_lauro_muller.html")
m

---
## Mapa cidade — Rio de Janeiro completo

Visão geral da cidade com todas as camadas sobrepostas: heatmap criminal 2020-2024, polígonos das 9 áreas FM, polígonos de domínio territorial ORCRIM (1.099 dentro do município) e todas as câmeras.

In [47]:
# ── Parâmetros cidade ─────────────────────────────────────────────────────────
RIO_BB = dict(lat_min=-23.10, lat_max=-22.74, lon_min=-43.80, lon_max=-43.09)

# Cores por facção ORCRIM
ORCRIM_COLOR = {
    "CV":     "#d73027",   # vermelho
    "Milícia":"#2166ac",   # azul
    "TCP":    "#f46d43",   # laranja
    "ADA":    "#762a83",   # roxo
}

# Cores das áreas FM por quartil de crimes (azul claro → azul escuro)
FM_CRIME_PALETTE = ["#c6dbef", "#6baed6", "#2171b5", "#084594"]

In [48]:
# ── 1. Crimes válidos (bbox Rio cidade) ───────────────────────────────────────
df_oc_all = pd.read_csv(
    SOURCES["Ocorrências Criminais"]["path"], encoding="utf-8", low_memory=False
)
df_crimes_rio = df_oc_all[
    df_oc_all["latitude"].between(RIO_BB["lat_min"], RIO_BB["lat_max"]) &
    df_oc_all["longitude"].between(RIO_BB["lon_min"], RIO_BB["lon_max"])
].copy()

print(f"Crimes válidos cidade: {len(df_crimes_rio):,} / {len(df_oc_all):,}")
print(df_crimes_rio["ano"].value_counts().sort_index())

# ── 2. Câmeras (todas) + lat/lon ──────────────────────────────────────────────
df_cam_all = pd.read_csv(
    SOURCES["Câmeras / Polígonos FM"]["path"], encoding="utf-8", low_memory=False
)
coords_all = df_cam_all["geometry"].str.extract(r"POINT \(([^ ]+) ([^\)]+)\)")
df_cam_all["lon"] = coords_all[0].astype(float)
df_cam_all["lat"] = coords_all[1].astype(float)
print(f"\nCâmeras totais: {len(df_cam_all)} em {df_cam_all['nome_area_fm'].nunique()} áreas")

Crimes válidos cidade: 115,318 / 115,354
ano
2020    26857
2021    25919
2022    21885
2023    18074
2024    22583
Name: count, dtype: int64

Câmeras totais: 985 em 9 áreas


In [49]:
# ── 3. Convex hulls das 9 áreas FM + contagem de crimes ──────────────────────
from shapely.geometry import MultiPoint, Point

crimes_arr_all = df_crimes_rio[["latitude", "longitude"]].to_numpy()

fm_areas = []
for area_name, grp in df_cam_all.groupby("nome_area_fm"):
    hull = MultiPoint([Point(r.lon, r.lat) for _, r in grp.iterrows()]).convex_hull
    # conta crimes dentro do hull expandido (buffer ~200 m)
    hull_buffered = hull.buffer(0.002)
    n = sum(
        1 for lat, lon in crimes_arr_all
        if hull_buffered.contains(Point(lon, lat))
    )
    hull_coords_fm = [(lat, lon) for lon, lat in hull.exterior.coords]
    fm_areas.append({"name": area_name, "hull": hull_coords_fm, "n_crimes": n})

fm_areas.sort(key=lambda x: x["n_crimes"])

# Atribui cor por quartil
q1, q2, q3 = np.percentile([a["n_crimes"] for a in fm_areas], [25, 50, 75])
def fm_color(n):
    if n <= q1: return FM_CRIME_PALETTE[0]
    if n <= q2: return FM_CRIME_PALETTE[1]
    if n <= q3: return FM_CRIME_PALETTE[2]
    return FM_CRIME_PALETTE[3]

for a in fm_areas:
    a["color"] = fm_color(a["n_crimes"])
    print(f"{a['n_crimes']:>6,} crimes | {a['name'][:55]}")

   371 crimes | Campo Grande: Estação de Trem - Calçadão
   423 crimes | Jardim de Alah
   640 crimes | Bangu: Calçadão - Bangu Shopping
   738 crimes | Rua Lauro Müller – Avenida General Severiano – Avenida 
 1,291 crimes | Metrô Botafogo - Rua São Clemente - Rua Voluntários da 
 1,518 crimes | Praia de Botafogo - Rua Marquês de Abrantes
 2,115 crimes | Estações São Francisco Xavier - Afonso Pena
 2,643 crimes | Rodoviária - Terminal Gentileza - Estação Leopoldina
 8,879 crimes | Presidente Vargas - Campo de Santana - Central do Brasi


In [50]:
# ── 4. Polígonos ORCRIM → GeoJSON FeatureCollection ─────────────────────────
import json
from shapely import wkt as shapely_wkt

df_dom_all = pd.read_csv(
    SOURCES["Domínio Territorial"]["path"], encoding="utf-8", low_memory=False
)

features = []
for _, row in df_dom_all.iterrows():
    try:
        geom = shapely_wkt.loads(row["geometria"])
        centroid = geom.centroid
        # filtra para cidade do Rio
        if not (RIO_BB["lat_min"] <= centroid.y <= RIO_BB["lat_max"] and
                RIO_BB["lon_min"] <= centroid.x <= RIO_BB["lon_max"]):
            continue
        features.append({
            "type": "Feature",
            "geometry": geom.__geo_interface__,
            "properties": {
                "nome": row["nome_territorio"],
                "orcrim": row["dominio_orcrim"],
                "color": ORCRIM_COLOR.get(row["dominio_orcrim"], "#999"),
            },
        })
    except Exception:
        continue

orcrim_geojson = {"type": "FeatureCollection", "features": features}
print(f"Polígonos ORCRIM no município: {len(features)}")
for fac, cor in ORCRIM_COLOR.items():
    n = sum(1 for f in features if f["properties"]["orcrim"] == fac)
    print(f"  {fac}: {n}")

Polígonos ORCRIM no município: 1099
  CV: 537
  Milícia: 387
  TCP: 151
  ADA: 24


In [51]:
# ── 5. Limites territoriais via OpenStreetMap ─────────────────────────────────
# Busca o limite municipal e os bairros do Rio via OSM (requer internet)
import osmnx as ox
import warnings

print("Buscando limite municipal...")
city_gdf = ox.geocode_to_gdf("Rio de Janeiro, Brazil")
city_geom = city_gdf.geometry.values[0]

print("Buscando bairros (admin_level 9 e 10)...")
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    b9  = ox.features_from_place("Rio de Janeiro, Brazil",
                                  tags={"boundary": "administrative", "admin_level": "9"})
    b10 = ox.features_from_place("Rio de Janeiro, Brazil",
                                  tags={"boundary": "administrative", "admin_level": "10"})

import pandas as pd
combined = pd.concat([b9, b10])
poly_all = combined[combined.geometry.geom_type.isin(["Polygon", "MultiPolygon"])].copy()

# Filtra centroides dentro do município (warning suprimido intencionalmente)
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    poly_all["in_city"] = poly_all.geometry.centroid.within(city_geom)

bairros_gdf = poly_all[poly_all["in_city"]].copy()
# Remove duplicatas pelo nome
bairros_gdf = bairros_gdf.drop_duplicates(subset=["name"])

print(f"Limite municipal: OK")
print(f"Bairros dentro do município: {len(bairros_gdf)}")

Buscando limite municipal...
Buscando bairros (admin_level 9 e 10)...
Limite municipal: OK
Bairros dentro do município: 165


In [ ]:
# ── 6. Montar mapa cidade ─────────────────────────────────────────────────────
import os

# Filtra crimes a partir de 2022
df_crimes_map = df_crimes_rio[df_crimes_rio["ano"] >= 2022].copy()
print(f"Crimes 2022-2024: {len(df_crimes_map):,} (de {len(df_crimes_rio):,} total)")

m_rio = folium.Map(
    location=[-22.92, -43.44],
    zoom_start=11,
    tiles="CartoDB dark_matter",
)

# --- Limite municipal (linha branca sólida) ------------------------------------
city_geojson = json.loads(city_gdf.to_json())
folium.GeoJson(
    city_geojson,
    name="Limite municipal",
    style_function=lambda _: {
        "color": "#ffffff",
        "weight": 2.5,
        "fillOpacity": 0,
    },
    tooltip="Município do Rio de Janeiro",
).add_to(m_rio)

# --- Bairros (linhas cinza claro finas, sem preenchimento) --------------------
bairros_geojson = json.loads(bairros_gdf[["geometry", "name"]].to_json())
folium.GeoJson(
    bairros_geojson,
    name="Bairros (limites)",
    style_function=lambda _: {
        "color": "#aaaaaa",
        "weight": 0.8,
        "dashArray": "4 3",
        "fillOpacity": 0,
    },
    tooltip=folium.GeoJsonTooltip(fields=["name"], aliases=["Bairro:"]),
    show=True,
).add_to(m_rio)

# --- ORCRIM: polígonos como GeoJSON (uma camada por facção para toggle) --------
for fac, cor in ORCRIM_COLOR.items():
    fac_features = [f for f in features if f["properties"]["orcrim"] == fac]
    if not fac_features:
        continue
    folium.GeoJson(
        {"type": "FeatureCollection", "features": fac_features},
        name=f"ORCRIM – {fac}",
        style_function=lambda feat, c=cor: {
            "fillColor": c,
            "color": c,
            "weight": 0.5,
            "fillOpacity": 0.35,
        },
        tooltip=folium.GeoJsonTooltip(
            fields=["orcrim", "nome"], aliases=["Facção:", "Território:"]
        ),
    ).add_to(m_rio)

# --- Heatmap de crimes 2022-2024 (subamostrado para performance no browser) ---
sample_size = min(30_000, len(df_crimes_map))
df_sample = df_crimes_map.sample(sample_size, random_state=42)
HeatMap(
    df_sample[["latitude", "longitude"]].values.tolist(),
    radius=14,
    blur=18,
    min_opacity=0.25,
    gradient={0.2: "#fff5f0", 0.5: "#fc8d59", 0.8: "#d7301f", 1.0: "#7f0000"},
    name="Heatmap crimes 2022-2024",
).add_to(m_rio)

# --- Áreas FM: convex hulls coloridos por crime count -------------------------
fm_layer = folium.FeatureGroup(name="Áreas FM (polígonos)", show=True)
for area in fm_areas:
    folium.Polygon(
        locations=area["hull"],
        color=area["color"],
        weight=2,
        fill=True,
        fill_color=area["color"],
        fill_opacity=0.25,
        tooltip=f"{area['name']}<br>{area['n_crimes']:,} crimes 2020-2024",
    ).add_to(fm_layer)
fm_layer.add_to(m_rio)

# --- Câmeras (desligado por padrão para não poluir visão geral) ---------------
cam_city_layer = folium.FeatureGroup(name="Câmeras FM", show=False)
for _, row in df_cam_all.iterrows():
    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=3,
        color="#74c476",
        fill=True,
        fill_color="#74c476",
        fill_opacity=0.8,
        weight=1,
        tooltip=f"Câmera | {row['nome_area_fm']}",
    ).add_to(cam_city_layer)
cam_city_layer.add_to(m_rio)

# --- Legenda ------------------------------------------------------------------
legend_rio = """
<div style="position:fixed;bottom:40px;right:20px;z-index:1000;
            background:rgba(20,20,20,0.88);color:white;
            padding:14px 18px;border-radius:8px;border:1px solid #555;
            font-family:Arial,sans-serif;font-size:12px;
            box-shadow:2px 2px 8px rgba(0,0,0,.5);">
  <b>Legenda</b><br><br>
  <span style="color:#ffffff">──</span> Limite municipal<br>
  <span style="color:#aaaaaa">╌╌</span> Bairros<br><br>
  <b>ORCRIM:</b><br>
  <span style="color:#d73027">&#9646;</span> CV &nbsp;
  <span style="color:#2166ac">&#9646;</span> Milícia &nbsp;
  <span style="color:#f46d43">&#9646;</span> TCP &nbsp;
  <span style="color:#762a83">&#9646;</span> ADA<br><br>
  <b>Áreas FM (intensidade criminal):</b><br>
  <span style="color:#c6dbef">&#9646;</span> Baixa &nbsp;
  <span style="color:#6baed6">&#9646;</span> Média-baixa<br>
  <span style="color:#2171b5">&#9646;</span> Média-alta &nbsp;
  <span style="color:#084594">&#9646;</span> Alta<br><br>
  <span style="color:#74c476">&#9679;</span> Câmera FM<br><br>
  <b>Crimes 2022-2024:</b><br>
  <span style="background:linear-gradient(to right,#fff5f0,#d7301f,#7f0000);
    display:inline-block;width:80px;height:8px;border-radius:3px"></span><br>
  Mínimo &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; Máximo
</div>
"""
m_rio.get_root().html.add_child(folium.Element(legend_rio))

title_rio = f"""
<div style="position:fixed;top:12px;left:50%;transform:translateX(-50%);
            z-index:1000;background:rgba(20,20,20,0.88);color:white;
            padding:8px 16px;border-radius:6px;border:1px solid #555;
            font-family:Arial,sans-serif;font-size:13px;
            box-shadow:2px 2px 6px rgba(0,0,0,.4);text-align:center;">
  <b>CompStat Rio — Mapa da cidade</b><br>
  <span style="color:#aaa">{len(df_crimes_map):,} ocorrências (2022-2024) · {len(features)} territórios ORCRIM · {len(bairros_gdf)} bairros</span>
</div>
"""
m_rio.get_root().html.add_child(folium.Element(title_rio))

folium.LayerControl(collapsed=False).add_to(m_rio)

# Salva HTML para abrir no browser
os.makedirs("../output", exist_ok=True)
m_rio.save("../output/mapa_rio.html")
print("Mapa salvo → abra: output/mapa_rio.html")
m_rio